# Building the training set: extra data and augmentation

The corpus is **unbalanced**: `angry` and `happy` have far fewer clips than `neutral` and `sad`. Two things were done to correct this, **on the training set only**:

1. **Extra clips** from a second corpus built from Tunisian YouTube videos (TV series, talk shows, podcasts): the `angry`, `happy` and `sad` clips were added to the training set.
2. **Data augmentation** of the `angry` and `happy` clips with three transformations: white noise, time stretching and pitch shifting.

| Set | Clips | angry | happy | neutral | sad |
|---|---|---|---|---|---|
| Training set after extra clips and augmentation | 5,587 | 1,494 | 1,347 | 1,465 | 1,281 |
| Test set (never augmented) | 1,025 | 228 | 108 | 394 | 295 |

Before these two steps the training set had 3,881 clips and the same imbalance as the test set (`neutral` and `sad` dominate, `happy` is the smallest class).

> **What this notebook is.** The three functions below are the ones printed in the thesis (figure "Augmentation techniques baseline code"); only the call to `librosa` was updated to the current API. The original notebook that produced the augmented clips was lost, and the **parameter values used in the original run were not recorded**: the values here are examples, to be tuned. The folder-level function at the end is new. The notebook runs without any data (it uses a synthetic signal for the demonstration).

## 1. Preparing the audio of the second corpus

The videos were downloaded and converted to audio, then cut into segments:

1. Video to audio with **FFmpeg**, then to **WAV, 16 kHz, mono** (the same format as the rest of the corpus), for example:
   `ffmpeg -i video.mp4 -vn -ac 1 -ar 16000 audio.wav`
2. The audio was cut **at silences**, so that no word is split, and so that each segment contains a single scene and a single speaker.
3. Each segment was listened to and annotated with an emotion, using a small annotation interface, and checked a second time.

This gave 2,429 segments (about 5 hours in total, from 12 videos), of which the `angry`, `happy` and `sad` segments were kept for training.

## 2. The three augmentation functions

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path


def add_white_noise(audio_signal, noise_percentage_factor):
    """Add Gaussian noise with the same standard deviation as the signal, scaled by the factor."""
    noise = np.random.normal(0, audio_signal.std(), audio_signal.size)
    augmented_audio_signal = audio_signal + noise * noise_percentage_factor
    return augmented_audio_signal


def time_stretch(audio_signal, time_stretch_rate):
    """Speed up (rate > 1) or slow down (rate < 1) the clip without changing its pitch."""
    return librosa.effects.time_stretch(audio_signal, rate=time_stretch_rate)


def pitch_scale(audio_signal, sr, num_semitones):
    """Raise or lower the pitch by a number of semitones without changing the speed."""
    return librosa.effects.pitch_shift(audio_signal, sr=sr, n_steps=num_semitones)

## 3. Demonstration on a synthetic signal

The cell below builds a one-second signal that mimics a voiced sound (a few harmonics with a slow amplitude variation) and applies each transformation. **The parameter values are examples.**

In [ ]:
sr = 16000
t = np.arange(sr) / sr
f0 = 180                                                   # fundamental frequency (Hz)
envelope = 0.6 + 0.4 * np.sin(2 * np.pi * 3 * t)           # slow amplitude variation
signal = envelope * sum(np.sin(2 * np.pi * f0 * k * t) / k for k in range(1, 6))
signal = (0.4 * signal / np.abs(signal).max()).astype("float32")

noisy = add_white_noise(signal, noise_percentage_factor=0.05)
stretched = time_stretch(signal, time_stretch_rate=1.2)
shifted = pitch_scale(signal, sr, num_semitones=2)

fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharey=True)
for ax, (name, y) in zip(axes, [("original", signal), ("white noise (factor 0.05)", noisy),
                                ("time stretch (rate 1.2)", stretched), ("pitch shift (+2 semitones)", shifted)]):
    ax.plot(np.arange(len(y)) / sr, y, linewidth=0.6)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()
print({"original": len(signal), "noise": len(noisy), "stretch": len(stretched), "pitch": len(shifted)})

## 4. Augmenting a folder of training clips

`augment_training_folder` creates augmented copies of the clips of the minority classes. Two safeguards come from the lessons of this project:

- it **refuses to run on the test folder**: augmented copies of test clips would inflate the scores, and the test set must stay the same real clips in every experiment;
- the augmented files get a name that shows where they come from, so that they can be traced (and, in a better split, kept in the same fold as their source clip).

Each target clip receives `copies` new clips, each made with one transformation chosen at random. The ranges below are examples.

In [ ]:
TARGET_EMOTIONS = ("angry", "happy")
NOISE_FACTORS = (0.02, 0.05, 0.1)
STRETCH_RATES = (0.8, 0.9, 1.1, 1.2)
SEMITONES = (-2, -1, 1, 2)


def augment_training_folder(train_dir, out_dir, label_of, copies=1, seed=0):
    """Write augmented copies of the target-emotion clips found in `train_dir`.

    label_of: dict file name -> emotion (as in the annotation file).
    Returns a list of (new file name, emotion) to add to the annotation file.
    """
    train_dir, out_dir = Path(train_dir), Path(out_dir)
    if any(part in ("val", "test") for part in train_dir.parts):
        raise ValueError("Refusing to augment a validation/test folder: augment the training set only.")
    rng = np.random.default_rng(seed)
    new_rows = []
    for wav in sorted(train_dir.rglob("*.wav")):
        emotion = label_of.get(wav.name)
        if emotion not in TARGET_EMOTIONS:
            continue
        y, sr = librosa.load(wav, sr=None, mono=True)
        for k in range(copies):
            kind = rng.choice(["noise", "stretch", "pitch"])
            if kind == "noise":
                y_aug = add_white_noise(y, float(rng.choice(NOISE_FACTORS)))
            elif kind == "stretch":
                y_aug = time_stretch(y, float(rng.choice(STRETCH_RATES)))
            else:
                y_aug = pitch_scale(y, sr, int(rng.choice(SEMITONES)))
            name = f"augmented_{wav.stem}_{kind}{k}.wav"
            target = out_dir / f"augmented_{emotion}"
            target.mkdir(parents=True, exist_ok=True)
            sf.write(target / name, y_aug, sr)
            new_rows.append((name, emotion))
    return new_rows

Example (uncomment and edit the paths): the augmented clips are written next to the training clips, and the returned rows are added to the annotation file.

```python
import pandas as pd
annotations = pd.read_excel("data/annotations.xlsx")
label_of = dict(zip(annotations["path"], annotations["label"]))
rows = augment_training_folder("data/train", "data/train", label_of, copies=1)
print(len(rows), "augmented clips written")
```

**What to keep in mind**

- Only the **training** set is augmented. After augmentation the training set is balanced (about 1,300 to 1,500 clips per class) while the test set keeps its natural, unbalanced distribution (108 `happy` against 394 `neutral`).
- Augmented copies are very close to their source clip. If the source clips are split by recording (see the README), keep each copy in the same fold as its source.
- Added noise and shifted pitch change what the model can learn about emotion itself (pitch is an emotion cue). Compare the models with and without augmentation, as in notebook 03.